This notebook ports the `mdp_tree_expansion_widget.jsx` lecture widget to Python. You get a direct, reproducible view of how the MDP lookahead tree grows as you vary depth, branching factor, discount $\gamma$, and reward pattern.

The tree alternates three node types to make the Bellman operators explicit:

- **State** $s$ (blue circle) — a `max` is taken over available actions; this is where $V(s)$ is formed.
- **Action** $a$ (green square) — immediate reward plus a discounted downstream expectation; this is where $Q(s,a)$ is formed.
- **Chance** (amber diamond) — expectation over transition outcomes.

The underlying Bellman backup equations are on the [parent lecture page](/aiml-common/lectures/mdp/bellman-optimality-backup); this notebook focuses on visualizing how the tree realizes them. The greedy policy path follows the best action at each state layer and the highest-value next state at each chance layer.

In [ ]:
%matplotlib inline

from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Tuple
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, Polygon
from matplotlib.patches import FancyArrowPatch

plt.rcParams["figure.dpi"] = 110

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)


def save_svg(fig, name: str) -> Path:
    """Save `fig` to images/<name>.svg and close it.

    Closing suppresses the inline PNG/SVG output so the generated MDX does not
    receive a duplicate auto-inserted <Frame><img/></Frame> block. The SVG on
    disk is referenced explicitly from the following markdown cell.
    """
    path = IMAGES_DIR / f"{name}.svg"
    fig.savefig(path, format="svg", bbox_inches="tight")
    plt.close(fig)
    return path

## Tree builder

The builder mirrors the JSX logic one-for-one. Leaf values are a deterministic function of level and branch index so repeated runs are visually stable, and action rewards depend on a configurable `reward_mode` (`goal-biased`, `uniform`, `risky`).

In [ ]:
@dataclass
class Node:
    node_id: int
    kind: str               # "state" | "action" | "chance"
    level: float            # integer for state, +0.25 for action, +0.5 for chance
    label: str
    value: float = 0.0
    reward: float = 0.0
    edge_prob: Optional[float] = None
    action_index: Optional[int] = None
    policy_choice: Optional[int] = None
    path_index: int = 0
    children: List["Node"] = field(default_factory=list)


def _leaf_value(level: int, branch_index: int, reward_mode: str) -> float:
    base = math.sin((level + 1) * 1.7 + branch_index * 0.9) * 4.0
    bonus = 3.0 if (reward_mode == "goal-biased" and branch_index % 3 == 0) else (
        -1.0 if reward_mode == "goal-biased" else 0.0
    )
    return base + bonus


def _reward_for_action(action_index: int, reward_mode: str) -> float:
    if reward_mode == "uniform":
        return -0.2
    if reward_mode == "goal-biased":
        return 0.8 if action_index == 0 else -0.3
    if reward_mode == "risky":
        return 1.6 if action_index == 0 else -0.1
    return 0.0


def _outcome_probabilities(k: int) -> List[float]:
    raw = [1.0 / (i + 1) for i in range(k)]
    s = sum(raw)
    return [v / s for v in raw]


def build_tree(
    depth: int,
    actions_per_state: int,
    outcomes_per_action: int,
    gamma: float,
    reward_mode: str = "goal-biased",
) -> Node:
    """Build an MDP lookahead tree and perform the Bellman backup.

    The returned root's `value` is V(s_root) under the greedy policy over the
    lookahead horizon of `depth` state transitions.
    """
    counter = {"n": 0}

    def next_id() -> int:
        counter["n"] += 1
        return counter["n"]

    def make_state(level: int, path_index: int = 0) -> Node:
        state = Node(
            node_id=next_id(),
            kind="state",
            level=float(level),
            label=f"s{level}",
            path_index=path_index,
        )

        if level >= depth:
            state.value = _leaf_value(level, path_index, reward_mode)
            return state

        action_children: List[Node] = []
        for a in range(actions_per_state):
            action = Node(
                node_id=next_id(),
                kind="action",
                level=level + 0.25,
                label=f"a{a}",
                reward=_reward_for_action(a, reward_mode),
                action_index=a,
                path_index=path_index * 10 + a,
            )
            chance = Node(
                node_id=next_id(),
                kind="chance",
                level=level + 0.5,
                label=f"P(-|a{a})",
                path_index=path_index * 10 + a,
            )
            probs = _outcome_probabilities(outcomes_per_action)
            for o in range(outcomes_per_action):
                next_state = make_state(level + 1, path_index * 100 + a * 10 + o)
                next_state.edge_prob = probs[o]
                chance.children.append(next_state)

            chance.value = sum(c.edge_prob * c.value for c in chance.children)
            action.children.append(chance)
            action.value = action.reward + gamma * chance.value
            action_children.append(action)

        state.children = action_children
        best_idx = max(range(len(action_children)), key=lambda i: action_children[i].value)
        state.policy_choice = best_idx
        state.value = action_children[best_idx].value
        return state

    return make_state(0, 1)


def count_nodes(root: Node) -> int:
    return 1 + sum(count_nodes(c) for c in root.children)


def count_leaves(root: Node) -> int:
    if not root.children:
        return 1
    return sum(count_leaves(c) for c in root.children)


def level_summary(root: Node) -> List[dict]:
    summary: List[dict] = []
    queue: List[Tuple[Node, int]] = [(root, 0)]
    while queue:
        node, d = queue.pop(0)
        while len(summary) <= d:
            summary.append({"state": 0, "action": 0, "chance": 0})
        summary[d][node.kind] += 1
        for child in node.children:
            child_d = d + 1 if child.kind == "state" else d
            queue.append((child, child_d))
    return summary

## Layout and rendering

Layout assigns an x-coordinate per level and a y-coordinate per node within that level, matching the JSX widget's left-to-right flow. The renderer draws states as circles, actions as rounded squares, and chance nodes as diamonds, with labels for rewards, probabilities, and backed-up values.

In [ ]:
STATE_COLOR = ("#dbeafe", "#2563eb")      # fill, stroke (blue)
ACTION_COLOR = ("#d1fae5", "#059669")     # emerald
CHANCE_COLOR = ("#fef3c7", "#d97706")     # amber

NODE_RADIUS = {"state": 0.35, "action": 0.30, "chance": 0.26}


def _level_key(node: Node) -> float:
    if node.kind == "state":
        return node.level * 3
    base = math.floor(node.level)
    if node.kind == "action":
        return base * 3 + 1
    return base * 3 + 2


def _collect(root: Node):
    levels: dict = {}
    edges: List[Tuple[Node, Node]] = []

    def visit(node: Node, parent: Optional[Node]) -> None:
        key = _level_key(node)
        levels.setdefault(key, []).append(node)
        if parent is not None:
            edges.append((parent, node))
        for c in node.children:
            visit(c, node)

    visit(root, None)
    sorted_levels = [v for _, v in sorted(levels.items(), key=lambda kv: kv[0])]
    return sorted_levels, edges


def _assign_positions(root: Node, width: float, height: float, margin_x: float, margin_y: float):
    sorted_levels, edges = _collect(root)
    pos: dict = {}
    level_count = len(sorted_levels)
    x_step = (width - 2 * margin_x) / max(level_count - 1, 1)

    for level_idx, nodes in enumerate(sorted_levels):
        nodes.sort(key=lambda n: n.path_index)
        count = len(nodes)
        y_step = (height - 2 * margin_y) / max(count - 1, 1) if count > 1 else 0
        for i, node in enumerate(nodes):
            x = margin_x + level_idx * x_step
            y = (height / 2) if count == 1 else (margin_y + i * y_step)
            pos[node.node_id] = (x, y)
    return pos, edges, sorted_levels


def _greedy_edges(root: Node):
    greedy = set()
    cur = root
    while cur is not None and cur.children:
        best_action = cur.children[cur.policy_choice]
        greedy.add((cur.node_id, best_action.node_id))
        chance = best_action.children[0] if best_action.children else None
        if chance is None:
            break
        greedy.add((best_action.node_id, chance.node_id))
        if not chance.children:
            break
        best_outcome = max(chance.children, key=lambda c: c.value)
        greedy.add((chance.node_id, best_outcome.node_id))
        cur = best_outcome
    return greedy


def _draw_node(ax, node: Node, x: float, y: float, show_values: bool, is_best_root_action: bool) -> None:
    r = NODE_RADIUS[node.kind]
    lw = 2.0 if is_best_root_action else 1.2
    if node.kind == "state":
        fill, stroke = STATE_COLOR
        ax.add_patch(Circle((x, y), r, facecolor=fill, edgecolor=stroke, linewidth=lw, zorder=3))
        glyph = "S"
    elif node.kind == "action":
        fill, stroke = ACTION_COLOR
        box = FancyBboxPatch(
            (x - r, y - r), 2 * r, 2 * r,
            boxstyle="round,pad=0.02,rounding_size=0.08",
            facecolor=fill, edgecolor=stroke, linewidth=lw, zorder=3,
        )
        ax.add_patch(box)
        glyph = "A"
    else:
        fill, stroke = CHANCE_COLOR
        diamond = Polygon(
            [(x, y - r), (x + r, y), (x, y + r), (x - r, y)],
            facecolor=fill, edgecolor=stroke, linewidth=lw, zorder=3,
        )
        ax.add_patch(diamond)
        glyph = "P"

    ax.text(x, y, glyph, ha="center", va="center", fontsize=7, fontweight="bold", zorder=4)

    if show_values:
        if node.kind == "state":
            ax.text(x + r + 0.08, y, f"V={node.value:.2f}", ha="left", va="center", fontsize=6.5, color="#0f172a", zorder=4)
        elif node.kind == "action":
            ax.text(x + r + 0.08, y + 0.12, f"Q={node.value:.2f}", ha="left", va="center", fontsize=6.5, color="#065f46", zorder=4)
            ax.text(x + r + 0.08, y - 0.12, f"r={node.reward:.2f}", ha="left", va="center", fontsize=6.0, color="#065f46", zorder=4)
        else:
            ax.text(x + r + 0.08, y, f"E={node.value:.2f}", ha="left", va="center", fontsize=6.5, color="#92400e", zorder=4)


def draw_tree(
    root: Node,
    ax=None,
    show_values: bool = True,
    highlight_greedy: bool = True,
    title: Optional[str] = None,
    figsize: Optional[Tuple[float, float]] = None,
):
    sorted_levels, _ = _collect(root)
    max_level_count = max(len(level) for level in sorted_levels)
    width = 14.0
    height = max(5.5, max_level_count * 0.55)

    if ax is None:
        inferred = figsize or (width, height)
        fig, ax = plt.subplots(figsize=inferred)
    else:
        fig = ax.figure

    positions, edges, sorted_levels = _assign_positions(root, width, height, margin_x=0.8, margin_y=0.6)
    greedy = _greedy_edges(root) if highlight_greedy else set()

    for idx, nodes in enumerate(sorted_levels):
        x, _ = positions[nodes[0].node_id]
        ax.plot([x, x], [0.1, height - 0.2], linestyle=":", color="#cbd5e1", linewidth=0.6, zorder=1)
        label = {"state": f"s@{int(nodes[0].level)}", "action": "a", "chance": "P"}[nodes[0].kind]
        ax.text(x, height - 0.05, label, ha="center", va="top", fontsize=7, color="#64748b", zorder=2)

    for parent, child in edges:
        x1, y1 = positions[parent.node_id]
        x2, y2 = positions[child.node_id]
        is_greedy = (parent.node_id, child.node_id) in greedy
        color = "#0f172a" if is_greedy else "#94a3b8"
        lw = 1.8 if is_greedy else 0.7
        arrow = FancyArrowPatch(
            (x1, y1), (x2, y2),
            arrowstyle="->",
            mutation_scale=7,
            connectionstyle="arc3,rad=0.08",
            color=color,
            linewidth=lw,
            zorder=2,
            alpha=0.9 if is_greedy else 0.45,
        )
        ax.add_patch(arrow)
        if child.edge_prob is not None and is_greedy:
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2 + 0.08
            ax.text(mx, my, f"p={child.edge_prob:.2f}", ha="center", va="bottom", fontsize=6, color="#0f172a", zorder=4)

    best_root_action_id = root.children[root.policy_choice].node_id if root.children else None
    for node_id, (x, y) in positions.items():
        node = _find(root, node_id)
        _draw_node(ax, node, x, y, show_values, is_best_root_action=(node_id == best_root_action_id))

    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_aspect("auto")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color="#0f172a", pad=4)
    return fig, ax


def _find(root: Node, node_id: int) -> Node:
    if root.node_id == node_id:
        return root
    for c in root.children:
        hit = _find(c, node_id)
        if hit is not None:
            return hit
    return None  # pragma: no cover

## A default expansion

The first tree uses planning depth 2 (fewer leaves so every label is readable), three actions per state, two stochastic outcomes per action, $\gamma = 0.9$, and a goal-biased reward pattern. The dark path is the greedy policy under the current backed-up values. Later cells show how width grows with depth.

In [ ]:
root = build_tree(depth=2, actions_per_state=3, outcomes_per_action=2, gamma=0.9, reward_mode="goal-biased")
fig, ax = draw_tree(root, title=f"Default tree — V(root) = {root.value:.2f}, total nodes = {count_nodes(root)}")
save_svg(fig, "default-tree")

<img src="images/default-tree.svg" alt="Default MDP lookahead tree at depth 2, three actions per state, two stochastic outcomes per action; greedy policy path highlighted in dark" />

*Default tree at depth 2 with the greedy policy path highlighted. Click to zoom.*

### Reading the diagram

- At **state** nodes the `max` operator acts.
- At **action** nodes the immediate reward combines with a discounted downstream expectation.
- At **chance** nodes the expectation operator acts over transition probabilities.
- The greedy path is defined at states but its value depends on chance-node averaging below.

## How width grows with depth and branching

The number of leaves scales as $(A \cdot O)^D$ where $A$ is actions per state, $O$ is outcomes per action, and $D$ is planning depth. Small increases in any of these explode width — this is the classic curse of dimensionality for tree-search planning.

In [ ]:
summary_root = build_tree(depth=4, actions_per_state=3, outcomes_per_action=2, gamma=0.9, reward_mode="goal-biased")
summary = level_summary(summary_root)
print(f"{'depth':>6} {'state':>8} {'action':>8} {'chance':>8}")
for d, row in enumerate(summary):
    print(f"{d:>6} {row['state']:>8} {row['action']:>8} {row['chance']:>8}")
print(f"\ntotal nodes: {count_nodes(summary_root)}, leaves: {count_leaves(summary_root)}")

In [ ]:
for d in [1, 2]:
    r = build_tree(depth=d, actions_per_state=3, outcomes_per_action=2, gamma=0.9, reward_mode="goal-biased")
    fig, ax = draw_tree(r, title=f"depth={d} — {count_nodes(r)} nodes, V(root)={r.value:.2f}")
    save_svg(fig, f"depth-{d}")

<img src="images/depth-1.svg" alt="MDP lookahead tree at depth 1 — single-step lookahead with 6 leaf states" />

*Depth 1 — a single state transition ahead.*

<img src="images/depth-2.svg" alt="MDP lookahead tree at depth 2 — two-step lookahead with 36 leaf states" />

*Depth 2 — two transitions ahead; leaves grow 6×.*

## Reward patterns change the greedy path

Three canned reward patterns illustrate how the Bellman backup reacts to per-action rewards:

- `uniform` — constant step cost for every action, so the greedy path reflects only downstream leaf values.
- `goal-biased` — action `a0` carries a positive immediate reward; the backup usually prefers it but not always.
- `risky` — action `a0` gives a much larger immediate reward (`+1.6`), often overwhelming downstream trade-offs.

In [ ]:
for mode in ["uniform", "goal-biased", "risky"]:
    r = build_tree(depth=2, actions_per_state=3, outcomes_per_action=2, gamma=0.9, reward_mode=mode)
    fig, ax = draw_tree(r, title=f"reward_mode={mode!r} — V(root)={r.value:.2f}, best action index = {r.policy_choice}")
    save_svg(fig, f"reward-{mode}")

<img src="images/reward-uniform.svg" alt="Uniform-reward MDP tree — all actions incur the same step cost, so the greedy path depends on downstream leaf values only" />

*`uniform` — every action costs the same; greedy path follows downstream values.*

<img src="images/reward-goal-biased.svg" alt="Goal-biased MDP tree — action a0 carries a positive reward; greedy root action typically prefers a0" />

*`goal-biased` — action `a0` carries a positive immediate reward.*

<img src="images/reward-risky.svg" alt="Risky-reward MDP tree — action a0 gives a large immediate reward (+1.6), usually dominating downstream trade-offs" />

*`risky` — action `a0`'s large immediate reward overwhelms downstream trade-offs.*

## Discount factor sweep

Varying $\gamma$ trades off immediate vs. downstream rewards. Low $\gamma$ values make the agent myopic, while $\gamma \to 1$ emphasizes the far future.

In [ ]:
gammas = [0.2, 0.5, 0.8, 0.99]
for g in gammas:
    r = build_tree(depth=3, actions_per_state=3, outcomes_per_action=2, gamma=g, reward_mode="goal-biased")
    print(f"gamma={g:>4}: V(root)={r.value:+.3f}, greedy root action = a{r.policy_choice}")

## Summary

- The lookahead tree alternates **state → action → chance → state**; the Bellman operators (max at states, expectation at chance nodes) sit exactly at those layers.
- Width grows as $(A\cdot O)^D$, which is why exact tree-search planning is only practical for short horizons or small branching factors.
- The greedy policy is defined at state nodes but its value depends on the expectation computed below the chance nodes — highlighting the path makes the backup flow visible from leaves to root.

Back to the lecture page: [Bellman Optimality Backup](/aiml-common/lectures/mdp/bellman-optimality-backup).